# 🚀 PULSE — Fine-tune Gemma 4 E2B on Venue Data

This notebook fine-tunes **Gemma 4 E2B** (2.3B effective params) on PULSE's scraped venue data using **QLoRA**.

**Requirements**: Colab T4 GPU (free tier works)

**Output**: A LoRA adapter + GGUF export ready for Ollama

**Tracking**: Training metrics logged to Weights & Biases

## 1. Install Dependencies

In [1]:
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q -U pyarrow datasets
!pip install -q torch trl peft accelerate bitsandbytes
!pip install -q huggingface_hub sentencepiece protobuf
!pip install -q wandb
print('✅ Dependencies installed (incl. wandb)')

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
✅ Dependencies installed (incl. wandb)


## 2. Login to Hugging Face & W&B

In [2]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [3]:
import wandb
wandb.login()  # paste your API key from https://wandb.ai/authorize
print('✅ W&B authenticated')

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: WARNING Invalid choice
wandb: Enter your choice:wandb: Enter your choice:wandb: WARNING Invalid choice
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ameth749 (yaatal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ W&B authenticated


## 3. Load Venue Data & Build Training Set

**Upload `nyc_seed.json`** using the Colab sidebar:
1. Click the 📁 folder icon on the left
2. Click the ⬆ upload button
3. Select `nyc_seed.json` from your PC
4. Then run this cell

In [5]:
import json, random, os

# Auto-detect nyc_seed.json location
CANDIDATES = [
    'nyc_seed.json',
    '/content/nyc_seed.json',
    '/content/drive/MyDrive/pulse/nyc_seed.json',
]

SEED_PATH = None
for p in CANDIDATES:
    if os.path.exists(p):
        SEED_PATH = p
        break

if SEED_PATH is None:
    raise FileNotFoundError(
        'nyc_seed.json not found! Upload it via the Colab sidebar: '
        'click the 📁 folder icon on the left → upload button → select the file.'
    )

with open(SEED_PATH) as f:
    venues = json.load(f)

print(f'✅ Loaded {len(venues)} venues from {SEED_PATH}')
print(f'Categories: {set(v["category"] for v in venues)}')

FileNotFoundError: nyc_seed.json not found! Upload it via the Colab sidebar: click the 📁 folder icon on the left → upload button → select the file.

In [ ]:
# Golden queries from PULSE eval suite
GOLDEN_QUERIES = [
    ('best tapas near me', 'Food'),
    ('Italian dinner tonight', 'Food'),
    ('yoga class tomorrow morning', 'Wellness'),
    ('craft beer bar', 'Nightlife'),
    ('brunch this weekend', 'Food'),
    ('contemporary art museum', 'Culture'),
    ('hidden gem restaurant', 'Food'),
    ('coffee shop to work from', 'Food'),
    ('rooftop bar', 'Nightlife'),
    ('japanese ramen', 'Food'),
    ('something fun for two', 'Fun'),
    ('late night food', 'Food'),
    ('wellness spa downtown', 'Wellness'),
    ('cheap eats', 'Food'),
    ('french bakery', 'Food'),
    ('live jazz tonight', 'Nightlife'),
    ('best pizza in Brooklyn', 'Food'),
    ('meditation studio', 'Wellness'),
    ('vintage clothing store', 'Shopping'),
    ('escape room for group', 'Fun'),
    ('sushi omakase', 'Food'),
    ('cocktail bar date night', 'Nightlife'),
    ('running trail near me', 'Nature'),
    ('bookshop with cafe', 'Shopping'),
    ('art gallery opening', 'Culture'),
]

In [ ]:
def build_training_examples(venues, queries, examples_per_query=8):
    """Generate (query, venues, ideal_summary) training triplets."""
    dataset = []
    by_cat = {}
    for v in venues:
        cat = v.get('category', 'Other')
        by_cat.setdefault(cat, []).append(v)

    for query_text, primary_cat in queries:
        pool = by_cat.get(primary_cat, venues)
        if len(pool) < 3:
            pool = venues

        for _ in range(examples_per_query):
            sampled = random.sample(pool, min(5, len(pool)))
            sampled.sort(key=lambda x: x.get('rating', 0), reverse=True)

            venue_lines = []
            for i, v in enumerate(sampled):
                name = v['name']
                cat = v['category']
                rating = v.get('rating', 'N/A')
                dist = v.get('distance', '?')
                tags = ', '.join(v.get('tags', [])[:3])
                venue_lines.append(f"{i+1}. {name} ({cat}, {rating}★, {dist}) [{tags}]")

            venue_block = '\n'.join(venue_lines)
            top = sampled[0]
            runner = sampled[1] if len(sampled) > 1 else sampled[0]

            summary = (
                f"{top['name']} stands out with a {top.get('rating','high')}★ rating "
                f"and is {top.get('distance','nearby')} away — perfect for {query_text.lower()}. "
                f"{runner['name']} is another solid pick if you want variety."
            )

            user_msg = (
                f"User query: {query_text}\n"
                f"Top venues:\n{venue_block}\n"
                f"Write a concise recommendation summary."
            )

            dataset.append({
                'messages': [
                    {'role': 'system', 'content': 'You summarize venue recommendations in 2 short sentences with practical tone.'},
                    {'role': 'user', 'content': user_msg},
                    {'role': 'assistant', 'content': summary},
                ]
            })

    random.shuffle(dataset)
    return dataset

random.seed(42)
train_data = build_training_examples(venues, GOLDEN_QUERIES, examples_per_query=8)
print(f'✅ Generated {len(train_data)} training examples')
print(f'\nSample:\n{json.dumps(train_data[0], indent=2)[:500]}')

✅ Generated 200 training examples

Sample:
{
  "messages": [
    {
      "role": "system",
      "content": "You summarize venue recommendations in 2 short sentences with practical tone."
    },
    {
      "role": "user",
      "content": "User query: best tapas near me\nTop venues:\n1. Sozai Japanese Restaurant (Izakaya Ramen) (Food, 4.9\u2605, 3.3 mi) [japanese_restaurant, ramen_restaurant, japanese_izakaya_restaurant]\n2. Osteria La Baia (Food, 4.9\u2605, 3.6 mi) [italian_restaurant, cocktail_bar, bar]\n3. Charlotte Patisserie (Food,


In [ ]:
# Save as JSONL for TRL
with open('pulse_venue_train.jsonl', 'w') as f:
    for row in train_data:
        f.write(json.dumps(row) + '\n')

from datasets import load_dataset
dataset = load_dataset('json', data_files='pulse_venue_train.jsonl', split='train')
print(f'✅ Dataset ready: {dataset}')

Generating train split: 0 examples [00:00, ? examples/s]

✅ Dataset ready: Dataset({
    features: ['messages'],
    num_rows: 200
})


## 4. Load Gemma 4 E2B with QLoRA

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = 'google/gemma-4-e2b-it'  # instruct-tuned variant

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False
print(f'✅ Model loaded: {MODEL_ID}')
print(f'   Memory: {model.get_memory_footprint() / 1e9:.2f} GB')

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

✅ Model loaded: google/gemma-4-e2b-it
   Memory: 6.70 GB


## 5. Configure LoRA & Train (with W&B)

In [ ]:
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
import wandb

# Initialize W&B run
wandb.init(
    project='pulse-gemma-finetune',
    name='gemma4-e2b-venue-qlora',
    config={
        'model': 'google/gemma-4-e2b-it',
        'method': 'QLoRA',
        'lora_r': 16,
        'lora_alpha': 32,
        'epochs': 3,
        'batch_size': 2,
        'grad_accum': 4,
        'lr': 2e-4,
        'train_examples': len(train_data),
    }
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)

training_args = SFTConfig(
    output_dir='./pulse-gemma-lora',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    fp16=True,  # Changed from bf16=True for T4 compatibility
    logging_steps=10,
    save_strategy='epoch',
    report_to='wandb',
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    peft_config=peft_config,
    processing_class=tokenizer,
    max_seq_length=512, # Moved here from SFTConfig
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'✅ Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')
print('🏋️ Starting training...')
trainer.train()
print('✅ Training complete!')

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


TypeError: SFTTrainer.__init__() got an unexpected keyword argument 'max_seq_length'

## 6. Save & Test

In [ ]:
# Save LoRA adapter
trainer.save_model('./pulse-gemma-lora/final')
tokenizer.save_pretrained('./pulse-gemma-lora/final')
print('✅ LoRA adapter saved')

In [ ]:
# Quick inference test
test_prompt = """User query: best tapas near me
Top venues:
1. AURA Tapas & Cocktail Bar (Nightlife, 4.9★, 1.4 mi)
2. Boqueria Soho (Food, 4.6★, 0.8 mi)
3. Toro NYC (Food, 4.5★, 1.1 mi)
Write a concise recommendation summary."""

messages = [
    {'role': 'system', 'content': 'You summarize venue recommendations in 2 short sentences with practical tone.'},
    {'role': 'user', 'content': test_prompt},
]

inputs = tokenizer.apply_chat_template(messages, return_tensors='pt', add_generation_prompt=True).to(model.device)
with torch.no_grad():
    outputs = model.generate(inputs, max_new_tokens=120, temperature=0.3, do_sample=True)

response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
print(f'Model response:\n{response}')

# Log sample to W&B
wandb.log({'sample_output': wandb.Html(f'<pre>{response}</pre>')})

## 7. Export to GGUF for Ollama

In [ ]:
# Merge LoRA into base model
from peft import AutoPeftModelForCausalLM

merged_model = AutoPeftModelForCausalLM.from_pretrained(
    './pulse-gemma-lora/final',
    device_map='auto',
    torch_dtype=torch.bfloat16,
)
merged_model = merged_model.merge_and_unload()
merged_model.save_pretrained('./pulse-gemma-merged')
tokenizer.save_pretrained('./pulse-gemma-merged')
print('✅ Merged model saved')

In [ ]:
# Convert to GGUF (requires llama.cpp)
!pip install -q gguf
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp 2>/dev/null || true
!python /content/llama.cpp/convert_hf_to_gguf.py ./pulse-gemma-merged --outtype q4_k_m --outfile pulse-gemma-e2b-q4.gguf
print('\n✅ GGUF exported: pulse-gemma-e2b-q4.gguf')

## 8. Finish & Download

After downloading the GGUF, register it with Ollama locally:
```bash
echo 'FROM ./pulse-gemma-e2b-q4.gguf' > Modelfile
ollama create pulse-gemma -f Modelfile
ollama run pulse-gemma
```

In [ ]:
# Finish W&B run
wandb.finish()
print('✅ W&B run finalized')

In [ ]:
# Upload LoRA adapter to HF Hub (optional)
HF_REPO = 'YOUR_USERNAME/pulse-gemma-e2b-venue-lora'  # change this

# from huggingface_hub import HfApi
# api = HfApi()
# api.upload_folder(folder_path='./pulse-gemma-lora/final', repo_id=HF_REPO, repo_type='model')
# print(f'✅ Uploaded to https://huggingface.co/{HF_REPO}')

In [ ]:
# Download GGUF file
from google.colab import files
files.download('pulse-gemma-e2b-q4.gguf')